In [1]:
import re
import json
import matplotlib.pyplot as plt
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_distances
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist, squareform
from sentence_transformers import SentenceTransformer
from kneed import KneeLocator
from transformers import pipeline
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import normalize
import seaborn as sns
from sklearn.manifold import TSNE
import nltk
from nltk.corpus import stopwords
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans, DBSCAN, SpectralClustering, AgglomerativeClustering

/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Improving the text representation 

Testing those 3 models: 
- paraphrase-multilingual-MiniLM-L12-v2
- distiluse-base-multilingual-cased-v2
- distilbert-base-german-cased 


If you go further down, every model can be run alone as well. The next code part includes all at once

In [ ]:

# CONFIGURATION
cq_path = "competency_questions_output/competency_questions_200_documents.txt"
embedding_models = {
    "MiniLM": "paraphrase-multilingual-MiniLM-L12-v2",
    "DistilUSE": "distiluse-base-multilingual-cased-v2",
    "GermanBERT": "distilbert-base-german-cased"
}
k = 220  

# Load questions
with open(cq_path, "r", encoding="utf-8") as f:
    content = f.read()
questions = re.findall(r'\*\*Frage:\*\*\s*(.+?)(?=\n\*\*Quelle:\*\*|\n\d+\.\Z)', content, re.DOTALL)
questions = list(dict.fromkeys(questions))

# Store results
results = []

def avg_intra_cluster_similarity(embeddings, labels):
    scores = []
    for cluster_id in set(labels):
        indices = [i for i, label in enumerate(labels) if label == cluster_id]
        if len(indices) < 2:
            continue
        cluster_embeds = embeddings[indices]
        sim_matrix = cosine_similarity(cluster_embeds)
        avg_sim = (np.sum(sim_matrix) - len(cluster_embeds)) / (len(cluster_embeds)**2 - len(cluster_embeds))
        scores.append(avg_sim)
    return np.mean(scores)

# Main loop over models
for name, model_name in embedding_models.items():
    print(f"\n🔄 Running model: {name}")
    model = SentenceTransformer(model_name)
    embeddings = model.encode(questions, show_progress_bar=True)
    
    X_emb_dense = squareform(pdist(embeddings, metric="cosine"))
    
    clustering = AgglomerativeClustering(metric='precomputed', linkage='average', n_clusters=k)
    labels = clustering.fit_predict(X_emb_dense)
    
    sil = silhouette_score(X_emb_dense, labels, metric="precomputed")
    db = davies_bouldin_score(embeddings, labels)
    ch = calinski_harabasz_score(embeddings, labels)
    avg_sim = avg_intra_cluster_similarity(np.array(embeddings), labels)
    
    results.append({
        "Model": name,
        "Silhouette": sil,
        "Davies-Bouldin": db,
        "Calinski-Harabasz": ch,
        "Avg_Intra_Similarity": avg_sim
    })
    
    # Save clustered data
    df = pd.DataFrame({
        "question": questions,
        "cluster": labels
    })
    df.to_csv(f"clustered_questions_{name}.csv", index=False)
    print(f"✅ Saved clustered_questions_{name}.csv")

# Display summary
summary = pd.DataFrame(results)
print("\n📊 Clustering Quality Comparison:")
print(summary.sort_values("Silhouette", ascending=False).round(3))


Silhouette Score: how well-separated and compact clusters are, higher = better
Davies-Bouldin Index: Avg. similiarity between each cluster and its most similar once, lower = better
Calinski: variance between clusters / within clusters, higher = better 
Avg. Intra similarity: avg. cosine similarity within each cluster, higher = better 


MiniLM has the best balance between separation and spread

YOU DO NOT HAVE TO NECESSARILY RUN THE NEXT 3 CODE SNIPPETS! 
Each of the 3 models are run separately there

In [ ]:
# Load questions
cq_path = "competency_questions_output/competency_questions_200_documents.txt"
with open(cq_path, "r", encoding="utf-8") as f:
    content = f.read()
questions = re.findall(r'\*\*Frage:\*\*\s*(.+?)(?=\n\*\*Quelle:\*\*|\n\d+\.\Z)', content, re.DOTALL)
questions = list(dict.fromkeys(questions))

# Embeddings
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embeddings = model.encode(questions)
X_emb_dense = squareform(pdist(embeddings, metric="cosine"))

# Auto-Tuning Clusteranzahl mit kombinierter Metrikbewertung
sil_scores, db_scores, ch_scores, ks = [], [], [], []
k_range = range(20, 251, 10)
all_labels = []

for k in k_range:
    clustering = AgglomerativeClustering(metric='precomputed', linkage='average', n_clusters=k)
    labels = clustering.fit_predict(X_emb_dense)
    sil = silhouette_score(X_emb_dense, labels, metric="precomputed")
    db = davies_bouldin_score(embeddings, labels)
    ch = calinski_harabasz_score(embeddings, labels)

    ks.append(k)
    sil_scores.append(sil)
    db_scores.append(db)
    ch_scores.append(ch)
    all_labels.append(labels)

# Normalisieren & kombinierte Bewertung
sil_norm = (sil_scores - np.mean(sil_scores)) / np.std(sil_scores)
db_norm = (db_scores - np.mean(db_scores)) / np.std(db_scores)
ch_norm = (ch_scores - np.mean(ch_scores)) / np.std(ch_scores)
combined_scores = sil_norm + ch_norm - db_norm

best_idx = np.argmax(combined_scores)
best_k = ks[best_idx]
best_labels = all_labels[best_idx]

# Plot
plt.figure(figsize=(12, 6))
plt.plot(ks, sil_scores, marker='o', label='Silhouette')
plt.plot(ks, db_scores, marker='s', label='Davies-Bouldin')
plt.plot(ks, ch_scores, marker='^', label='Calinski-Harabasz')
plt.title("Cluster-Metriken über verschiedene Clusteranzahlen")
plt.xlabel("Anzahl Cluster (k)")
plt.ylabel("Metrik-Wert")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"\n✅ Beste Clusteranzahl (kombiniert bewertet): k={best_k}")


# Deutsche Stopwords laden
nltk.download('stopwords')
german_stopwords = stopwords.words('german')

# Synonymliste für juristische Begriffe (später mit DATEV abstimmen)
synonym_map = {
    "vergütung": ["entlohnung", "honorar", "bezahlung"],
    "kündigung": ["beendigung", "auflösung"],
    "vertrag": ["vereinbarung"],
}

def apply_synonyms(texts, mapping):
    updated = []
    for t in texts:
        for target, syns in mapping.items():
            for s in syns:
                t = t.replace(s, target)
        updated.append(t)
    return updated

# Synonyme anwenden
questions_syn = apply_synonyms(questions, synonym_map)

# TF-IDF berechnen
vectorizer = TfidfVectorizer(stop_words=german_stopwords)
X_tfidf = vectorizer.fit_transform(questions_syn)
feature_names = np.array(vectorizer.get_feature_names_out())

# Keywords pro Cluster (nutzt best_labels)
print("\n🔎 Top TF-IDF Keywords pro Cluster:")
for cluster_id in sorted(set(best_labels)):
    cluster_indices = [i for i, label in enumerate(best_labels) if label == cluster_id]
    cluster_texts = [questions_syn[i] for i in cluster_indices]
    cluster_matrix = vectorizer.transform(cluster_texts).mean(axis=0).A1
    top_keywords = feature_names[cluster_matrix.argsort()[-5:][::-1]]
    print(f"\n📁 Cluster {cluster_id} (n={len(cluster_indices)}):")
    print("Top Keywords:", ", ".join(top_keywords))



In [ ]:
# Load questions
cq_path = "competency_questions_output/competency_questions_200_documents.txt"
with open(cq_path, "r", encoding="utf-8") as f:
    content = f.read()
questions = re.findall(r'\*\*Frage:\*\*\s*(.+?)(?=\n\*\*Quelle:\*\*|\n\d+\.\Z)', content, re.DOTALL)
questions = list(dict.fromkeys(questions))

# Embeddings
model = SentenceTransformer("distiluse-base-multilingual-cased-v2")
embeddings = model.encode(questions)
X_emb_dense = squareform(pdist(embeddings, metric="cosine"))

# Auto-Tuning Clusteranzahl mit kombinierter Metrikbewertung
sil_scores, db_scores, ch_scores, ks = [], [], [], []
k_range = range(20, 251, 10)
all_labels = []

for k in k_range:
    clustering = AgglomerativeClustering(metric='precomputed', linkage='average', n_clusters=k)
    labels = clustering.fit_predict(X_emb_dense)
    sil = silhouette_score(X_emb_dense, labels, metric="precomputed")
    db = davies_bouldin_score(embeddings, labels)
    ch = calinski_harabasz_score(embeddings, labels)

    ks.append(k)
    sil_scores.append(sil)
    db_scores.append(db)
    ch_scores.append(ch)
    all_labels.append(labels)

# Normalisieren & kombinierte Bewertung
sil_norm = (sil_scores - np.mean(sil_scores)) / np.std(sil_scores)
db_norm = (db_scores - np.mean(db_scores)) / np.std(db_scores)
ch_norm = (ch_scores - np.mean(ch_scores)) / np.std(ch_scores)
combined_scores = sil_norm + ch_norm - db_norm

best_idx = np.argmax(combined_scores)
best_k = ks[best_idx]
best_labels = all_labels[best_idx]

# Plot
plt.figure(figsize=(12, 6))
plt.plot(ks, sil_scores, marker='o', label='Silhouette')
plt.plot(ks, db_scores, marker='s', label='Davies-Bouldin')
plt.plot(ks, ch_scores, marker='^', label='Calinski-Harabasz')
plt.title("Cluster-Metriken über verschiedene Clusteranzahlen")
plt.xlabel("Anzahl Cluster (k)")
plt.ylabel("Metrik-Wert")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"\n✅ Beste Clusteranzahl (kombiniert bewertet): k={best_k}")


# Deutsche Stopwords laden
nltk.download('stopwords')
german_stopwords = stopwords.words('german')

# Synonymliste für juristische Begriffe (später mit DATEV abstimmen)
synonym_map = {
    "vergütung": ["entlohnung", "honorar", "bezahlung"],
    "kündigung": ["beendigung", "auflösung"],
    "vertrag": ["vereinbarung"],
}

def apply_synonyms(texts, mapping):
    updated = []
    for t in texts:
        for target, syns in mapping.items():
            for s in syns:
                t = t.replace(s, target)
        updated.append(t)
    return updated

# Synonyme anwenden
questions_syn = apply_synonyms(questions, synonym_map)

# TF-IDF berechnen
vectorizer = TfidfVectorizer(stop_words=german_stopwords)
X_tfidf = vectorizer.fit_transform(questions_syn)
feature_names = np.array(vectorizer.get_feature_names_out())

# Keywords pro Cluster (nutzt best_labels)
print("\n🔎 Top TF-IDF Keywords pro Cluster:")
for cluster_id in sorted(set(best_labels)):
    cluster_indices = [i for i, label in enumerate(best_labels) if label == cluster_id]
    cluster_texts = [questions_syn[i] for i in cluster_indices]
    cluster_matrix = vectorizer.transform(cluster_texts).mean(axis=0).A1
    top_keywords = feature_names[cluster_matrix.argsort()[-5:][::-1]]
    print(f"\n📁 Cluster {cluster_id} (n={len(cluster_indices)}):")
    print("Top Keywords:", ", ".join(top_keywords))



In [ ]:
# Load questions
cq_path = "competency_questions_output/competency_questions_200_documents.txt"
with open(cq_path, "r", encoding="utf-8") as f:
    content = f.read()
questions = re.findall(r'\*\*Frage:\*\*\s*(.+?)(?=\n\*\*Quelle:\*\*|\n\d+\.\Z)', content, re.DOTALL)
questions = list(dict.fromkeys(questions))

# Embeddings
model = SentenceTransformer("distilbert-base-german-cased")
embeddings = model.encode(questions)
X_emb_dense = squareform(pdist(embeddings, metric="cosine"))

# Auto-Tuning Clusteranzahl mit kombinierter Metrikbewertung
sil_scores, db_scores, ch_scores, ks = [], [], [], []
k_range = range(20, 251, 10)
all_labels = []

for k in k_range:
    clustering = AgglomerativeClustering(metric='precomputed', linkage='average', n_clusters=k)
    labels = clustering.fit_predict(X_emb_dense)
    sil = silhouette_score(X_emb_dense, labels, metric="precomputed")
    db = davies_bouldin_score(embeddings, labels)
    ch = calinski_harabasz_score(embeddings, labels)

    ks.append(k)
    sil_scores.append(sil)
    db_scores.append(db)
    ch_scores.append(ch)
    all_labels.append(labels)

# Normalisieren & kombinierte Bewertung
sil_norm = (sil_scores - np.mean(sil_scores)) / np.std(sil_scores)
db_norm = (db_scores - np.mean(db_scores)) / np.std(db_scores)
ch_norm = (ch_scores - np.mean(ch_scores)) / np.std(ch_scores)
combined_scores = sil_norm + ch_norm - db_norm

best_idx = np.argmax(combined_scores)
best_k = ks[best_idx]
best_labels = all_labels[best_idx]

# Plot
plt.figure(figsize=(12, 6))
plt.plot(ks, sil_scores, marker='o', label='Silhouette')
plt.plot(ks, db_scores, marker='s', label='Davies-Bouldin')
plt.plot(ks, ch_scores, marker='^', label='Calinski-Harabasz')
plt.title("Cluster-Metriken über verschiedene Clusteranzahlen")
plt.xlabel("Anzahl Cluster (k)")
plt.ylabel("Metrik-Wert")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"\n✅ Beste Clusteranzahl (kombiniert bewertet): k={best_k}")


# Deutsche Stopwords laden
nltk.download('stopwords')
german_stopwords = stopwords.words('german')

# Synonymliste für juristische Begriffe (später mit DATEV abstimmen)
synonym_map = {
    "vergütung": ["entlohnung", "honorar", "bezahlung"],
    "kündigung": ["beendigung", "auflösung"],
    "vertrag": ["vereinbarung"],
}

def apply_synonyms(texts, mapping):
    updated = []
    for t in texts:
        for target, syns in mapping.items():
            for s in syns:
                t = t.replace(s, target)
        updated.append(t)
    return updated

# Synonyme anwenden
questions_syn = apply_synonyms(questions, synonym_map)

# TF-IDF berechnen
vectorizer = TfidfVectorizer(stop_words=german_stopwords)
X_tfidf = vectorizer.fit_transform(questions_syn)
feature_names = np.array(vectorizer.get_feature_names_out())

# Keywords pro Cluster (nutzt best_labels)
print("\n🔎 Top TF-IDF Keywords pro Cluster:")
for cluster_id in sorted(set(best_labels)):
    cluster_indices = [i for i, label in enumerate(best_labels) if label == cluster_id]
    cluster_texts = [questions_syn[i] for i in cluster_indices]
    cluster_matrix = vectorizer.transform(cluster_texts).mean(axis=0).A1
    top_keywords = feature_names[cluster_matrix.argsort()[-5:][::-1]]
    print(f"\n📁 Cluster {cluster_id} (n={len(cluster_indices)}):")
    print("Top Keywords:", ", ".join(top_keywords))



In [4]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# CONFIGURATION
cq_path = "competency_questions_output/competency_questions_200_documents.txt"
embedding_models = {
    "MiniLM": "paraphrase-multilingual-MiniLM-L12-v2",
    "DistilUSE": "distiluse-base-multilingual-cased-v2",
    "GermanBERT": "distilbert-base-german-cased"
}
k = 220  # You can change this or run your own k-selection logic

# Load questions
with open(cq_path, "r", encoding="utf-8") as f:
    content = f.read()
questions = re.findall(r'\*\*Frage:\*\*\s*(.+?)(?=\n\*\*Quelle:\*\*|\n\d+\.\Z)', content, re.DOTALL)
questions = list(dict.fromkeys(questions))

# Store results
results = []

def avg_intra_cluster_similarity(embeddings, labels):
    scores = []
    for cluster_id in set(labels):
        indices = [i for i, label in enumerate(labels) if label == cluster_id]
        if len(indices) < 2:
            continue
        cluster_embeds = embeddings[indices]
        sim_matrix = cosine_similarity(cluster_embeds)
        avg_sim = (np.sum(sim_matrix) - len(cluster_embeds)) / (len(cluster_embeds)**2 - len(cluster_embeds))
        scores.append(avg_sim)
    return np.mean(scores)

# Main loop over models
for name, model_name in embedding_models.items():
    print(f"\n🔄 Running model: {name}")
    model = SentenceTransformer(model_name)
    embeddings = model.encode(questions, show_progress_bar=True)
    
    X_emb_dense = squareform(pdist(embeddings, metric="cosine"))
    
    clustering = AgglomerativeClustering(metric='precomputed', linkage='average', n_clusters=k)
    labels = clustering.fit_predict(X_emb_dense)
    
    sil = silhouette_score(X_emb_dense, labels, metric="precomputed")
    db = davies_bouldin_score(embeddings, labels)
    ch = calinski_harabasz_score(embeddings, labels)
    avg_sim = avg_intra_cluster_similarity(np.array(embeddings), labels)
    
    results.append({
        "Model": name,
        "Silhouette": sil,
        "Davies-Bouldin": db,
        "Calinski-Harabasz": ch,
        "Avg_Intra_Similarity": avg_sim
    })
    
    # Save clustered data
    df = pd.DataFrame({
        "question": questions,
        "cluster": labels
    })
    df.to_csv(f"clustered_questions_{name}.csv", index=False)
    print(f"✅ Saved clustered_questions_{name}.csv")

# Display summary
summary = pd.DataFrame(results)
print("\n📊 Clustering Quality Comparison:")
print(summary.sort_values("Silhouette", ascending=False).round(3))



🔄 Running model: MiniLM


Batches: 100%|██████████| 54/54 [00:43<00:00,  1.24it/s]


✅ Saved clustered_questions_MiniLM.csv

🔄 Running model: DistilUSE


Batches: 100%|██████████| 54/54 [01:26<00:00,  1.60s/it]
No sentence-transformers model found with name distilbert-base-german-cased. Creating a new one with mean pooling.


✅ Saved clustered_questions_DistilUSE.csv

🔄 Running model: GermanBERT


Batches: 100%|██████████| 54/54 [01:09<00:00,  1.28s/it]


✅ Saved clustered_questions_GermanBERT.csv

📊 Clustering Quality Comparison:
        Model  Silhouette  Davies-Bouldin  Calinski-Harabasz  \
0      MiniLM       0.171           1.423              9.210   
1   DistilUSE       0.138           1.541              6.777   
2  GermanBERT       0.095           1.614             17.581   

   Avg_Intra_Similarity  
0                 0.743  
1                 0.677  
2                 0.961  



MiniML outperfroms all models. 



HERE COMES THE IMPORTANT PART: ALL SENTENCE EMEBEDDINGS ARE TESTED AGAINNST EACH OTHER, USING HDBSCAN !

Try the  models using  HDBSCAN instead of agglomerative clustering


In [2]:
# %pip install umap-learn sentence-transformers hdbscan

import re
import numpy as np
import pandas as pd
import nltk
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sentence_transformers import SentenceTransformer
import hdbscan
import umap

nltk.download('stopwords')

# Load questions
with open("competency_questions_output/competency_questions_200_documents.txt", "r", encoding="utf-8") as f:
    content = f.read()
questions = list(dict.fromkeys(
    re.findall(r'\*\*Frage:\*\*\s*(.+?)(?=\n\*\*Quelle:\*\*|\n\d+\.\Z)', content, re.DOTALL)
))

# Models to compare
models = {
    "MiniLM": "paraphrase-multilingual-MiniLM-L12-v2",
    "GBERT": "deepset/gbert-base",
    "DistilUSE": "distiluse-base-multilingual-cased-v2",
    "GermanBERT": "distilbert-base-german-cased"
}

# Result collection
results = []

# Processing loop
for name, model_name in models.items():
    print(f"\n🔄 Encoding with: {name}")
    model = SentenceTransformer(model_name)
    embeddings = model.encode(
        questions,
        batch_size=16,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    # UMAP projection
    reducer = umap.UMAP(n_neighbors=15, n_components=10, metric='cosine', random_state=42)
    reduced = reducer.fit_transform(embeddings)

    # HDBSCAN clustering
    clusterer = hdbscan.HDBSCAN(min_cluster_size=5, metric='euclidean')
    labels = clusterer.fit_predict(reduced)

    # Filter out noise
    valid_indices = [i for i, l in enumerate(labels) if l != -1]
    if len(set(labels)) <= 1 or len(valid_indices) < 2:
        results.append({"Model": name, "Silhouette": None, "DB": None, "CH": None, "NumClusters": 0})
        continue

    # Evaluate clustering
    sil = silhouette_score([reduced[i] for i in valid_indices], [labels[i] for i in valid_indices])
    db = davies_bouldin_score([reduced[i] for i in valid_indices], [labels[i] for i in valid_indices])
    ch = calinski_harabasz_score([reduced[i] for i in valid_indices], [labels[i] for i in valid_indices])

    results.append({
        "Model": name,
        "Silhouette": sil,
        "DB": db,
        "CH": ch,
        "NumClusters": len(set(labels)) - (1 if -1 in labels else 0)
    })

# Show results
df = pd.DataFrame(results)
print("\n📊 Model Comparison Summary:")
print(df.round(3))


[nltk_data] Downloading package stopwords to /Users/jule/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



🔄 Encoding with: MiniLM


Batches: 100%|██████████| 107/107 [01:08<00:00,  1.56it/s]
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
No sentence-transformers model found with name deepset/gbert-base. Creating a new one with


🔄 Encoding with: GBERT


Batches: 100%|██████████| 107/107 [03:03<00:00,  1.72s/it]
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


...

Batches: 100%|██████████| 107/107 [02:06<00:00,  1.18s/it]
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
No sentence-transformers model found with name distilbert-base-german-cased. Creating a ne


🔄 Encoding with: GermanBERT


Batches: 100%|██████████| 107/107 [01:36<00:00,  1.11it/s]
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(



📊 Model Comparison Summary:
        Model  Silhouette     DB         CH  NumClusters
0      MiniLM       0.546  0.515   1841.890           75
1       GBERT       0.382  0.660    182.055           61
2   DistilUSE       0.523  0.542    850.781           57
3  GermanBERT       0.473  0.534  31006.310            5


/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Improve clustering by trying different clustering algorithm

Try the following: 
Agglomerative
KMeans
HDBSCAN 
Spectral Clustering

In [ ]:
%pip install hdbscan

from sklearn.cluster import KMeans, DBSCAN, SpectralClustering, AgglomerativeClustering
import hdbscan

# Load questions
cq_path = "competency_questions_output/competency_questions_200_documents.txt"
with open(cq_path, "r", encoding="utf-8") as f:
    content = f.read()
questions = list(dict.fromkeys(
    re.findall(r'\*\*Frage:\*\*\s*(.+?)(?=\n\*\*Quelle:\*\*|\n\d+\.\Z)', content, re.DOTALL)
))

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embeddings = model.encode(questions, show_progress_bar=True)


def evaluate_clustering(name, labels, embeddings):
    if len(set(labels)) <= 1 or -1 in set(labels) and len(set(labels)) <= 2:
        return {"Model": name, "Silhouette": None, "Davies-Bouldin": None, "CH": None, "IntraSim": None}
    sil = silhouette_score(embeddings, labels)
    db = davies_bouldin_score(embeddings, labels)
    ch = calinski_harabasz_score(embeddings, labels)
    sim = avg_intra_cluster_similarity(np.array(embeddings), labels)
    return {"Model": name, "Silhouette": sil, "Davies-Bouldin": db, "CH": ch, "IntraSim": sim}

def avg_intra_cluster_similarity(embeddings, labels):
    scores = []
    for cluster_id in set(labels):
        if cluster_id == -1:  # skip noise
            continue
        indices = [i for i, label in enumerate(labels) if label == cluster_id]
        if len(indices) < 2:
            continue
        cluster_embeds = embeddings[indices]
        sim_matrix = cosine_similarity(cluster_embeds)
        avg_sim = (np.sum(sim_matrix) - len(cluster_embeds)) / (len(cluster_embeds)**2 - len(cluster_embeds))
        scores.append(avg_sim)
    return np.mean(scores) if scores else None

results = []

# Agglomerative 
X_dense = squareform(pdist(embeddings, metric="cosine"))
agg = AgglomerativeClustering(metric='precomputed', linkage='average', n_clusters=80)
results.append(evaluate_clustering("Agglomerative", agg.fit_predict(X_dense), embeddings))

# KMeans
kmeans = KMeans(n_clusters=80, random_state=42, n_init="auto")
results.append(evaluate_clustering("KMeans", kmeans.fit_predict(embeddings), embeddings))

# HDBSCAN
hdb = hdbscan.HDBSCAN(min_cluster_size=5, metric="euclidean")
results.append(evaluate_clustering("HDBSCAN", hdb.fit_predict(embeddings), embeddings))

# Spectral Clustering
spec = SpectralClustering(n_clusters=80, affinity='nearest_neighbors', assign_labels='kmeans', random_state=42)
results.append(evaluate_clustering("Spectral", spec.fit_predict(embeddings), embeddings))

# Print Results
df = pd.DataFrame(results)
print("\n📊 Clustering Algorithm Comparison (MiniLM Embeddings):")
print(df.round(3))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


Batches: 100%|██████████| 54/54 [00:40<00:00,  1.35it/s]
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(



📊 Clustering Algorithm Comparison (MiniLM Embeddings):
           Model  Silhouette  Davies-Bouldin      CH  IntraSim
0  Agglomerative       0.055           1.618  13.169     0.695
1         KMeans       0.072           2.413  18.794     0.697
2         DBSCAN         NaN             NaN     NaN       NaN
3        HDBSCAN       0.147           3.812   8.476     0.619
4       Spectral       0.062           2.227  17.393     0.725


Best clustering option: HDBSCAN 

Advantages: no need to specify k; can identify outliers; good scalability and interpretability

For HDBSCAN, manually review all questions that have a noise of -1 to avoid forcing them into a random group 

In [14]:
%pip install hdbscan

import pandas as pd
import hdbscan

# Run HDBSCAN
clusterer = hdbscan.HDBSCAN(min_cluster_size=5, metric='euclidean')
labels = clusterer.fit_predict(embeddings)

# Extract noise points
noise_indices = [i for i, label in enumerate(labels) if label == -1]
noise_questions = [questions[i] for i in noise_indices]

# Optional: get outlier scores (confidence of being noise)
outlier_scores = clusterer.outlier_scores_[noise_indices]

# Create DataFrame for review
noise_df = pd.DataFrame({
    "Question": noise_questions,
    "OutlierScore": outlier_scores
}).sort_values("OutlierScore", ascending=False)

# Save to CSV for manual review
noise_df.to_csv("noise_questions_review.csv", index=False)

print(f"✅ Exported {len(noise_df)} noise questions to noise_questions_review.csv")


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


✅ Exported 24 noise questions to noise_questions_review.csv


I manually reviewed the questions that were identified as noise (label = -1), see "noise_questions_review.csv". 

I think that all noise-labeled questions are still important and should be kept - no really unimportant ones. 

So now reintegrating the noise questions by reassign them to the nearest existing cluster based on cosine similarity.

In [15]:

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

# Isolate non-noise data
non_noise_indices = [i for i, label in enumerate(labels) if label != -1]
non_noise_embeddings = np.array(embeddings)[non_noise_indices]
non_noise_labels = np.array(labels)[non_noise_indices]

# Extract noise questions and embeddings
noise_indices = [i for i, label in enumerate(labels) if label == -1]
noise_questions = [questions[i] for i in noise_indices]
noise_embeddings = np.array(embeddings)[noise_indices]

# Assign each noise question to the most similar cluster
reassigned_labels = []

for i, noise_embed in enumerate(noise_embeddings):
    # Compute cosine similarity to all non-noise embeddings
    similarities = cosine_similarity([noise_embed], non_noise_embeddings)[0]
    best_match_index = np.argmax(similarities)
    best_cluster_label = non_noise_labels[best_match_index]
    reassigned_labels.append(best_cluster_label)

# Merge reassigned labels into the full label list
final_labels = labels.copy()
for idx, new_label in zip(noise_indices, reassigned_labels):
    final_labels[idx] = new_label

print(f"✅ Reassigned {len(noise_indices)} noise questions to nearest existing clusters.")


✅ Reassigned 24 noise questions to nearest existing clusters.


Further refine the clusters by giving each cluster a label with TF IDF and remove stop words

In [16]:
nltk.download('stopwords')
stop_words = set(nltk.corpus.stopwords.words('german'))

# Load & extract questions
cq_path = "competency_questions_output/competency_questions_200_documents.txt"
with open(cq_path, "r", encoding="utf-8") as f:
    content = f.read()

questions = list(dict.fromkeys(
    re.findall(r'\*\*Frage:\*\*\s*(.+?)(?=\n\*\*Quelle:\*\*|\n\d+\.\Z)', content, re.DOTALL)
))

# Generate embeddings using MiniLM
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embeddings = model.encode(questions, show_progress_bar=True)

# Initial clustering using HDBSCAN
clusterer = hdbscan.HDBSCAN(min_cluster_size=5, metric='euclidean')
labels = clusterer.fit_predict(embeddings)

# Manually reviewed noise (-1) are kept and reassigned
non_noise_indices = [i for i, l in enumerate(labels) if l != -1]
noise_indices = [i for i, l in enumerate(labels) if l == -1]
non_noise_embeddings = np.array(embeddings)[non_noise_indices]
non_noise_labels = np.array(labels)[non_noise_indices]
noise_embeddings = np.array(embeddings)[noise_indices]

reassigned_labels = []
for i, embed in enumerate(noise_embeddings):
    sims = cosine_similarity([embed], non_noise_embeddings)[0]
    best_idx = np.argmax(sims)
    best_label = non_noise_labels[best_idx]
    reassigned_labels.append(best_label)

# Merge reassigned noise labels back
final_labels = labels.copy()
for idx, new_label in zip(noise_indices, reassigned_labels):
    final_labels[idx] = new_label

# Merge similar clusters based on centroid similarity
# Compute centroids
cluster_dict = {}
for i, label in enumerate(final_labels):
    cluster_dict.setdefault(label, []).append(embeddings[i])

centroids = {label: np.mean(vecs, axis=0) for label, vecs in cluster_dict.items()}
labels_list = list(centroids.keys())
centroid_matrix = np.array([centroids[l] for l in labels_list])
similarity_matrix = cosine_similarity(centroid_matrix)

# Merge clusters with similarity > 0.95
threshold = 0.95
label_map = {l: l for l in labels_list}

for i in range(len(labels_list)):
    for j in range(i + 1, len(labels_list)):
        if similarity_matrix[i, j] > threshold:
            src = labels_list[j]
            tgt = labels_list[i]
            for k, v in label_map.items():
                if v == src:
                    label_map[k] = tgt
            label_map[src] = tgt

# Apply new merged cluster labels
merged_labels = [label_map[l] for l in final_labels]

# Clean questions
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def remove_stopwords(text):
    return " ".join([word for word in text.split() if word not in stop_words])

cleaned_questions = [remove_stopwords(clean_text(q)) for q in questions]

# Label clusters using top TF-IDF keywords
df = pd.DataFrame({
    "Question": questions,
    "CleanQuestion": cleaned_questions,
    "MergedCluster": merged_labels
})

cluster_labels = sorted(df["MergedCluster"].unique())
cluster_keywords = {}

for label in cluster_labels:
    group = df[df["MergedCluster"] == label]["CleanQuestion"]
    if len(group) < 2:
        cluster_keywords[label] = [f"Cluster {label}"]
        continue
    vectorizer = TfidfVectorizer(max_features=5)
    tfidf = vectorizer.fit_transform(group)
    top_words = vectorizer.get_feature_names_out()
    cluster_keywords[label] = top_words.tolist()

df["ClusterLabel"] = df["MergedCluster"].apply(lambda x: ", ".join(cluster_keywords.get(x, [f"Cluster {x}"])))

# Export final result
df[["Question", "ClusterLabel", "MergedCluster"]].to_csv("final_cluster_named.csv", index=False)
print("✅ All steps complete. Output saved to final_cluster_named.csv")

[nltk_data] Downloading package stopwords to /Users/jule/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
Batches: 100%|██████████| 54/54 [00:45<00:00,  1.19it/s]
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


✅ All steps complete. Output saved to final_cluster_named.csv
